# I import the modules that I need for parsing the formulas
# and solving the system of equations.

In [ ]:
import re
import numpy as np

from scipy.linalg import null_space
from fractions import Fraction
from math import gcd
from functools import reduce

# Part B – Stoichiometry and Reaction Balancing

In this part, I use the chemical formulas from Part A to balance chemical reactions.

My main idea is to first count the atoms of each element in every molecule.
Then, I represent the reaction as a matrix and use linear algebra to find
the stoichiometric coefficients.

I also check the result in two different ways:
- by checking that the number of atoms of every element is conserved;
- by checking that the total molecular mass is conserved.



# B1. Parsing chemical formulas

Before balancing a reaction, I need to know how many atoms of each element
are present in every molecule.

For example, H2O should give 2 hydrogen atoms and 1 oxygen atom.

For this part, I do not need to handle parentheses or hydration water,
because the project says that these cases do not need to be considered
for reaction balancing.

In [ ]:
def parse_molecule(formula):
    # I first check that the input is actually a non-empty string.
    if not isinstance(formula, str) or not formula:
        raise ValueError("The chemical formula must be a non-empty string.")

    # An element starts with an uppercase letter and can have
    # one lowercase letter after it. A number is optional.
    pattern = r'([A-Z][a-z]?)(\d*)'
    matches = re.findall(pattern, formula)

    # I reconstruct the formula from the matches.
    # This lets me check that the whole formula was understood.
    reconstructed = "".join(
        element + number for element, number in matches
    )

    if reconstructed != formula:
        raise ValueError(f"Invalid chemical formula: {formula}")

    atom_counts = {}

    # I go through every element found in the formula
    # and store its number of atoms in a dictionary.
    for element, number in matches:
        count = int(number) if number else 1

        atom_counts[element] = atom_counts.get(element, 0) + count

    return atom_counts

In [ ]:
# I test the parser with different molecules to make sure
# that it works with both one- and two-letter element symbols.

test_formulas = [
    "H2O",
    "CO2",
    "Fe2O3",
    "C3H8",
    "C2H5OH",
    "NH3"
]

for formula in test_formulas:
    print(f"{formula}: {parse_molecule(formula)}")


H2O: {'H': 2, 'O': 1}
CO2: {'C': 1, 'O': 2}
Fe2O3: {'Fe': 2, 'O': 3}
C3H8: {'C': 3, 'H': 8}
C2H5OH: {'C': 2, 'H': 6, 'O': 1}
NH3: {'N': 1, 'H': 3}


## B2. Creating the balance matrix

I represent the reaction as a matrix.

For example, for:

H2 + O2 -> H2O

the matrix is:

        H2   O2   H2O
H        2    0   -2
O        0    2   -1

I use positive values for the reactants and negative values for the products.

This means that if c is the vector containing the unknown coefficients,
a balanced reaction satisfies:

A * c = 0

The negative sign for the products allows the conservation of each element
to be represented by the same equation.

In [ ]:
def create_balance_matrix(reactants, products):

    # I check that reactants and products are given as lists.
    if not isinstance(reactants, list) or not isinstance(products, list):
        raise ValueError("Reactants and products must be lists.")

    # A reaction needs at least one reactant and one product.
    if len(reactants) == 0 or len(products) == 0:
        raise ValueError(
            "A reaction must contain at least one reactant and one product."
        )

    # I combine both sides because each molecule will correspond
    # to one column in the matrix.
    formulas = reactants + products

    # I convert every molecular formula into a dictionary of atoms.
    parsed_molecules = [
        parse_molecule(formula)
        for formula in formulas
    ]

    # I collect all the different elements appearing in the reaction.
    elements = sorted({
        element
        for molecule in parsed_molecules
        for element in molecule
    })

    # Rows correspond to elements and columns correspond to molecules.
    matrix = np.zeros((len(elements), len(formulas)))

    for row, element in enumerate(elements):

        for column, molecule in enumerate(parsed_molecules):

            atom_count = molecule.get(element, 0)

            # Reactants are positive and products are negative.
            if column < len(reactants):
                matrix[row, column] = atom_count
            else:
                matrix[row, column] = -atom_count

    return matrix, elements

reactants = ["H2", "O2"]
products = ["H2O"]

matrix, elements = create_balance_matrix(
    reactants,
    products
)

print("Elements:", elements)
print("Balance matrix:")
print(matrix)

Elements: ['H', 'O']
Balance matrix:
[[ 2.  0. -2.]
 [ 0.  2. -1.]]


# B3. Finding the stoichiometric coefficients

At this point I have a matrix describing the conservation of each element.

I searched for a linear algebra method that could solve this type of problem.
The null space is useful here because I need to find a non-zero vector c such
that:

A * c = 0

I use `null_space` from scipy.linalg to find this vector.

The result is usually given as floating-point numbers, so I then convert
the values into the smallest possible integer coefficients.

In [ ]:
def integerize_coefficients(vector):

    # I convert the vector to a numpy array so that I can work with it.
    vector = np.asarray(vector, dtype=float)

    # Very small values can appear because of numerical precision.
    # I consider them to be zero.
    vector[np.abs(vector) < 1e-12] = 0

    if np.allclose(vector, 0):
        raise ValueError(
            "The input vector is a zero vector; cannot integerize."
        )

    # The null-space vector can have all its signs reversed.
    # I choose the positive version when necessary.
    if np.all(vector < 0):
        vector = -vector

    # Stoichiometric coefficients should be positive.
    if np.any(vector <= 0):
        raise ValueError(
            "A positive stoichiometric solution could not be found."
        )

    # The numerical values are converted into fractions.
    # This helps recover simple integer ratios such as 1/2.
    fractions = [
        Fraction(float(value)).limit_denominator(10000)
        for value in vector
    ]

    # I calculate the least common multiple of all denominators.
    def lcm(a, b):
        return abs(a * b) // gcd(a, b)

    common_denominator = reduce(
        lcm,
        [fraction.denominator for fraction in fractions]
    )

    # I multiply all fractions by the common denominator.
    integer_values = [
        fraction.numerator *
        (common_denominator // fraction.denominator)
        for fraction in fractions
    ]

    # Finally, I divide by the greatest common divisor
    # to obtain the smallest integer coefficients.
    common_divisor = reduce(gcd, integer_values)

    integer_values = [
        value // common_divisor
        for value in integer_values
    ]

    return integer_values


# B4. Checking the number of atoms

I do not want to assume that the coefficients returned by the calculation
are correct, so I added another check.

For each element, I calculate the total number of atoms on the reactant side
and on the product side.

If the two dictionaries are identical, every element is conserved.

In [ ]:
def verify_atom_balance(reactants, products, coefficients):

    number_of_reactants = len(reactants)

    # I separate the coefficients belonging to each side of the reaction.
    reactant_coefficients = coefficients[:number_of_reactants]
    product_coefficients = coefficients[number_of_reactants:]

    reactant_atoms = {}

    # I count the total number of atoms on the reactant side.
    for formula, coeff in zip(reactants, reactant_coefficients):

        atoms = parse_molecule(formula)

        for element, count in atoms.items():
            reactant_atoms[element] = (
                reactant_atoms.get(element, 0)
                + count * coeff
            )

    product_atoms = {}

    # I do the same calculation for the products.
    for formula, coeff in zip(products, product_coefficients):

        atoms = parse_molecule(formula)

        for element, count in atoms.items():
            product_atoms[element] = (
                product_atoms.get(element, 0)
                + count * coeff
            )

    return reactant_atoms == product_atoms

# B5. Reaction balancing function

I can now combine the previous steps into one function.

The function first creates the matrix, then finds its null space,
converts the numerical solution into integers and finally verifies
that the resulting coefficients really conserve all the atoms.

In [ ]:
def balance_reaction(reactants, products):

    # First I create the matrix representing the reaction.
    matrix, elements = create_balance_matrix(
        reactants,
        products
    )

    # I search for the vectors satisfying A*c = 0.
    solution_space = null_space(matrix)

    if solution_space.size == 0:
        raise ValueError(
            "No solution exists for the given reaction."
        )

    # For a normal chemical reaction, I expect one independent
    # stoichiometric solution.
    if solution_space.shape[1] > 1:
        raise ValueError(
            "Multiple independent solutions exist; "
            "cannot determine a unique stoichiometric solution."
        )

    # I take the first vector of the null space.
    numerical_solution = solution_space[:, 0]

    # The null-space values are converted to integers.
    coefficients = integerize_coefficients(
        numerical_solution
    )

    # I check the result before returning it.
    if not verify_atom_balance(
        reactants,
        products,
        coefficients
    ):
        raise ValueError(
            "The computed coefficients do not balance the reaction."
        )

    return coefficients

# B6. Testing the reaction balancer

I test the function with three different reactions.

The first one is a simple reaction with two reactants and one product.
The second contains iron, and the third is the combustion of propane.

I use different examples to check that the function is not only working
for the example used to construct the matrix.

In [ ]:
# Reaction 1
reactants_1 = ["H2", "O2"]
products_1 = ["H2O"]

coefficients_1 = balance_reaction(
    reactants_1,
    products_1
)

print("Reaction 1:")
print(coefficients_1)

# Reaction 2
reactants_2 = ["Fe", "O2"]
products_2 = ["Fe2O3"]

coefficients_2 = balance_reaction(
    reactants_2,
    products_2
)

print("Reaction 2:")
print(coefficients_2)

# Reaction 3
reactants_3 = ["C3H8", "O2"]
products_3 = ["CO2", "H2O"]

coefficients_3 = balance_reaction(
    reactants_3,
    products_3
)

print("Reaction 3:")
print(coefficients_3)

Reaction 1:
[2, 1, 2]
Reaction 2:
[4, 3, 2]
Reaction 3:
[1, 5, 3, 4]


# B7. Mass conservation

After balancing the reactions using the number of atoms, I also want to
check the conservation of mass.

I use the molecular mass calculator developed in Part A.

For each molecule, I multiply its molecular mass by its stoichiometric
coefficient and then compare the total mass on both sides.

Because molecular masses are represented using floating-point numbers,
I use a small tolerance instead of requiring the two values to be
exactly identical.

In [ ]:
def calculate_total_mass(molecules):

    total_mass = 0.0

    # molecules is a dictionary where the key is the formula
    # and the value is its stoichiometric coefficient.
    for formula, coefficient in molecules.items():

        mass = molecular_weight_calculator

        total_mass += mass * coefficient

    return total_mass

In [ ]:
def check_mass_conservation(
    reactants,
    products,
    tolerance=1e-6
):

    reactant_mass = calculate_total_mass(reactants)
    product_mass = calculate_total_mass(products)

    # Because the masses are floating-point numbers,
    # I compare them using a tolerance.
    return abs(reactant_mass - product_mass) <= tolerance


In [ ]:
def display_mass_check(
    reactants,
    products,
    tolerance=1e-6
):

    reactant_mass = calculate_total_mass(reactants)
    product_mass = calculate_total_mass(products)

    balanced = check_mass_conservation(
        reactants,
        products,
        tolerance
    )

    print(f"Reactants total mass: {reactant_mass:.3f} u")
    print(f"Products total mass: {product_mass:.3f} u")

    if balanced:
        print("Mass is conserved within the specified tolerance.")
    else:
        print("Mass is NOT conserved within the specified tolerance.")

    return balanced

In [ ]:
def reaction_to_dictionaries(
    reactants,
    products,
    coefficients
):

    number_of_reactants = len(reactants)

    # I split the coefficient list into the reactant
    # and product coefficients.
    reactant_coefficients = coefficients[:number_of_reactants]
    product_coefficients = coefficients[number_of_reactants:]

    reactants_dict = dict(
        zip(reactants, reactant_coefficients)
    )

    products_dict = dict(
        zip(products, product_coefficients)
    )

    return reactants_dict, products_dict

# B9. Mass conservation tests

I now use the coefficients found by the reaction balancer to create
dictionaries containing each molecule and its coefficient.

I then calculate the total mass on both sides of each reaction.

The purpose of this test is to check the result using a different
property of the reaction: conservation of mass.

In [ ]:
# Reaction 1

reactants_1_dict, products_1_dict = reaction_to_dictionaries(
    reactants_1,
    products_1,
    coefficients_1
)

print("Reaction 1")
print("Reactants:", reactants_1_dict)
print("Products:", products_1_dict)

display_mass_check(
    reactants_1_dict,
    products_1_dict
)

# Reaction 2

reactants_2_dict, products_2_dict = reaction_to_dictionaries(
    reactants_2,
    products_2,
    coefficients_2
)

print("Reaction 2")
print("Reactants:", reactants_2_dict)
print("Products:", products_2_dict)

display_mass_check(
    reactants_2_dict,
    products_2_dict
)

# Reaction 3

reactants_3_dict, products_3_dict = reaction_to_dictionaries(
    reactants_3,
    products_3,
    coefficients_3
)

print("Reaction 3")
print("Reactants:", reactants_3_dict)
print("Products:", products_3_dict)

display_mass_check(
    reactants_3_dict,
    products_3_dict
)

Reaction 1
Reactants: {'H2': 2, 'O2': 1}
Products: {'H2O': 2}


NameError: name 'molecular_weight_calculator' is not defined

# Failed experiment

I also tested the mass conservation function with incorrect coefficients.

For the combustion of propane, I used 4 molecules of O2 instead of 5:

equation = "C3H8 + 4O2 -> 3CO2 + 4H2O"

I expected the function to return False because the reaction is not balanced.

This was useful to check that the mass conservation function is actually
performing a calculation instead of simply assuming that the reaction
is balanced.

In [ ]:
wrong_reactants = {
    "C3H8": 1,
    "O2": 4
}

wrong_products = {
    "CO2": 3,
    "H2O": 4
}

result = check_mass_conservation(
    wrong_reactants,
    wrong_products
)

print("Mass conservation result:", result)

if result:
    print("Mass is conserved.")
else:
    print("Mass is NOT conserved.")


NameError: name 'molecular_weight_calculator' is not defined

# Additional tests

Finally, I use assertions to automatically check two cases.

The first reaction has the correct coefficients and should conserve mass.

The second one deliberately uses incorrect coefficients and should not
conserve mass.

In [ ]:
# Correct reaction:
# 2H2 + O2 -> 2H2O

assert check_mass_conservation(
    {"H2": 2, "O2": 1},
    {"H2O": 2}
), "Mass is not conserved for 2H2 + O2 -> 2H2O"


# Incorrect reaction:
# C3H8 + 4O2 -> 3CO2 + 4H2O

assert not check_mass_conservation(
    {"C3H8": 1, "O2": 4},
    {"CO2": 3, "H2O": 4}
), "The incorrect reaction was detected as balanced."

print("All mass conservation tests passed.")


NameError: name 'molecular_weight_calculator' is not defined